# ViHSD - Vietnamese Hate Speech Detection with PhoBERT

# 1. Library & Setup

In [1]:
# Cài đặt thư viện: pyvi (tách từ tiếng Việt), emoji (xử lý biểu tượng cảm xúc), transformers và accelerate
!pip install -q pyvi emoji "transformers>=4.38.0" "accelerate>=0.33.0"

In [2]:
import os
import re
import random
import unicodedata

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sn

import torch
import torch.nn as nn
from torch.utils.data import Dataset

import transformers
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments,
    DataCollatorWithPadding
)
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    f1_score,
    precision_recall_fscore_support,
    accuracy_score
)

import emoji
from pyvi import ViTokenizer

# Cố định Random Seed 42
SEED = 42

def seed_everything(seed=SEED):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True

seed_everything(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Thiết bị tính toán: {device} | Seed: {SEED}")

EVAL_STRATEGY_PARAM = "eval_strategy" if hasattr(transformers.TrainingArguments, "eval_strategy") else "evaluation_strategy"

OUTPUT_DIR = os.path.abspath("./outputs/seed42_teencode_35")
CHECKPOINT_DIR = os.path.abspath("./checkpoints/seed42_teencode_35")
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

print(f"📁 Thư mục kết quả:     {OUTPUT_DIR}")
print(f"📁 Thư mục checkpoints: {CHECKPOINT_DIR}")


# 2. Data loader
Bộ dữ liệu chuẩn gồm 33,400 bình luận Facebook phân chia theo tỷ lệ 72:8:20 (Train: 24,048; Dev: 2,672; Test: 6,680).

In [3]:
# Nạp dữ liệu ViHSD trực tiếp từ thư mục ./data/
train = pd.read_csv("./data/train.csv")
dev   = pd.read_csv("./data/dev.csv")
test  = pd.read_csv("./data/test.csv")

# Chuẩn hóa cột id
for df in [train, dev, test]:
    if 'id' not in df.columns:
        df['id'] = range(len(df))

print(f"Số lượng mẫu Train: {len(train)}")
print(f"Số lượng mẫu Dev:   {len(dev)}")
print(f"Số lượng mẫu Test:  {len(test)}")


# 3. Pre-processing & Word Segmentation
Cải tiến quy trình tiền xử lý theo Pipeline:
- **Chuẩn hóa Unicode NFC:** Đồng nhất các biến thể gõ dấu tiếng Việt.
- **Xóa Emoji & Icon:** Ngăn ngừa phân mảnh từ và lỗi xử lý.
- **Xóa URL & Email:** Tránh subword rác làm tràn độ dài.
- **Chuẩn hóa Teencode & Viết tắt (Tran et al., 2022):** Ánh xạ từ viết tắt (`ko -> không`, `dc -> được`, `vs -> với`, `đ.ụ -> đụ`,...).
- **Không xóa stopwords:** Kế thừa phát hiện khoa học của Luu et al. (2024), việc giữ nguyên từ dừng giúp mô hình bảo toàn ngữ cảnh self-attention.
- **Tách từ tiếng Việt qua PyVi:** Nối từ ghép bằng dấu gạch dưới `_` (`ngôn_từ`, `thù_ghét`) khớp với không gian BPE của PhoBERT.
- **Chống rò rỉ dữ liệu (Data Leakage):** Khử trùng lặp nội bộ Train và loại bỏ câu Train trùng với Dev/Test; bảo toàn tuyệt đối 100% mẫu Dev và Test.

In [5]:
# Nạp từ điển teencode_35 trực tiếp từ ./data/teencode_35.json
import json
import re
import unicodedata

with open("./data/teencode_35.json", "r", encoding="utf-8") as f:
    TEENCODE_DICT = json.load(f)

print(f"✅ Đã nạp thành công {len(TEENCODE_DICT)} từ vựng từ ./data/teencode_35.json")

# Biên dịch các cụm teencode nhiều từ
_MULTI_WORD_PATTERNS = [
    (re.compile(r"\b" + re.escape(k) + r"\b", re.IGNORECASE), v)
    for k, v in TEENCODE_DICT.items() if " " in k
]

def normalize_teencode_text(text: str) -> str:
    if not isinstance(text, str) or not text:
        return ""
    for pattern, replacement in _MULTI_WORD_PATTERNS:
        text = pattern.sub(replacement, text)
    tokens = re.findall(r"\w+|[^\w\s]", text, re.UNICODE)
    normalized_tokens = [TEENCODE_DICT.get(t.lower(), t) for t in tokens]
    return " ".join(normalized_tokens)

def preprocess_text(text, tokenized=True):
    if not isinstance(text, str):
        return ""
    # 1. Chuẩn hóa Unicode NFC
    text = unicodedata.normalize('NFC', text)
    # 2. Xóa biểu tượng cảm xúc (Emoji)
    text = emoji.replace_emoji(text, replace='')
    # 3. Xóa URL & Email
    text = re.sub(r"http\S+|www\S+", " ", text)
    text = re.sub(r"\S+@\S+", " ", text)
    # 4. Gỡ dấu chèn ngụy trang từ ngữ (đ.ụ -> đụ, đ-ĩ -> đĩ)
    text = re.sub(r"(\b[a-zA-ZđĐáàảãạăắằẳẵặâấầẩẫậéèẻẽẹêếềểễệíìỉĩịóòỏõọôốồổỗộơớờởỡợúùủũụưứừửữựýỳỷỹỵ])[\.\-_/\*]+([a-zA-ZđĐáàảãạăắằẳẵặâấầẩẫậéèẻẽẹêếềểễệíìỉĩịóòỏõọôốồổỗộơớờởỡợúùủũụưứừửữựýỳỷỹỵ])", r"\1\2", text)
    # 5. Chuẩn hóa Teencode
    text = normalize_teencode_text(text)
    # 6. Chuẩn hóa khoảng trắng
    text = re.sub(r"\s+", " ", text).strip()
    # 7. Tách từ tiếng Việt bằng ViTokenizer
    if tokenized and len(text) > 0:
        text = ViTokenizer.tokenize(text)
    return text

print("Đang thực hiện tiền xử lý và tách từ...")
train['clean_text'] = train['free_text'].apply(preprocess_text)
dev['clean_text']   = dev['free_text'].apply(preprocess_text)
test['clean_text']  = test['free_text'].apply(preprocess_text)

# Xử lý rỗng an toàn
dev['clean_text'] = dev['clean_text'].apply(lambda t: "trống" if len(t.strip()) == 0 else t)
test['clean_text'] = test['clean_text'].apply(lambda t: "trống" if len(t.strip()) == 0 else t)

# Khử trùng lặp Train
train_clean = train[train['clean_text'].str.strip().str.len() > 0].copy()
train_clean = train_clean.drop_duplicates(subset=['clean_text'], keep='first')

# Chống rò rỉ dữ liệu (Data Leakage)
dev_test_texts = set(dev['clean_text']).union(set(test['clean_text']))
init_len = len(train_clean)
train_clean = train_clean[~train_clean['clean_text'].isin(dev_test_texts)].reset_index(drop=True)
print(f"✅ Train sạch sau lọc: {len(train_clean)} mẫu (loại bỏ {init_len - len(train_clean)} mẫu trùng lặp/rò rỉ).")

X_train = train_clean['clean_text'].tolist()
y_train = train_clean['label_id'].values

X_dev = dev['clean_text'].tolist()
y_dev = dev['label_id'].values

X_test = test['clean_text'].tolist()
y_test = test['label_id'].values


# 4. Class Weight Calculation
Tính toán trọng số cân bằng lớp (Balanced Class Weight) trên tập Train sạch theo công thức:
$$w_c = \frac{N}{3 \times N_c}$$

In [6]:
# Tính toán trọng số lớp
class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(y_train),
    y=y_train
)
class_weights_tensor = torch.tensor(class_weights, dtype=torch.float).to(device)

LABEL_NAMES = {0: "CLEAN", 1: "OFFENSIVE", 2: "HATE"}
print("=" * 60)
print("TRỌNG SỐ LỚP CÂN BẰNG (BALANCED CLASS WEIGHTS)")
print("=" * 60)
for c, w in enumerate(class_weights):
    cnt = sum(y_train == c)
    print(f"Lớp {c} ({LABEL_NAMES[c]:<9}): {cnt:>5} mẫu ({cnt/len(y_train)*100:>5.2f}%) | Trọng số w_{c} = {w:.4f}")
print("=" * 60)

# 5. Dataset, Tokenizer & Custom Trainer
- Sử dụng mô hình nền tảng tối ưu: **`vinai/phobert-base-v2`** với `max_length = 128` và `DataCollatorWithPadding` (đệm động theo batch).
- Kế thừa lớp `BuildDataset` từ tác giả bài báo.
- Xây dựng lớp `WeightedLossTrainer` kế thừa `Trainer` của tác giả để hỗ trợ truyền trọng số lớp vào hàm mất mát `nn.CrossEntropyLoss(weight=...)`.

In [7]:
# Nạp Tokenizer
MODEL_NAME = "vinai/phobert-base-v2"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=False)

# Lớp Dataset (Kế thừa BuildDataset của tác giả)
class BuildDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=128):
        self.texts = list(texts)
        self.labels = list(labels)
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __getitem__(self, idx):
        text = str(self.texts[idx])
        label = int(self.labels[idx])
        encoding = self.tokenizer(
            text,
            truncation=True,
            max_length=self.max_len,
            padding=False,  # Để DataCollatorWithPadding đệm động theo batch
            return_attention_mask=True
        )
        return {
            'input_ids': encoding['input_ids'],
            'attention_mask': encoding['attention_mask'],
            'labels': torch.tensor(label, dtype=torch.long)
        }

    def __len__(self):
        return len(self.labels)

train_dataset = BuildDataset(X_train, y_train, tokenizer, max_len=128)
dev_dataset   = BuildDataset(X_dev, y_dev, tokenizer, max_len=128)
test_dataset  = BuildDataset(X_test, y_test, tokenizer, max_len=128)

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

# Custom Trainer hỗ trợ Class Weight
class WeightedLossTrainer(Trainer):
    def __init__(self, class_weights=None, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.class_weights = class_weights

    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None, **kwargs):
        labels = inputs.get("labels")
        outputs = model(**inputs)
        logits = outputs.get("logits")

        if self.class_weights is not None:
            loss_fct = nn.CrossEntropyLoss(weight=self.class_weights.to(model.device))
        else:
            loss_fct = nn.CrossEntropyLoss()

        loss = loss_fct(logits.view(-1, self.model.config.num_labels), labels.view(-1))
        return (loss, outputs) if return_outputs else loss

# Hàm tính Macro F1
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    mf1 = f1_score(labels, preds, average='macro', zero_division=0)
    wf1 = f1_score(labels, preds, average='weighted', zero_division=0)
    acc = accuracy_score(labels, preds)
    p, r, f1, _ = precision_recall_fscore_support(labels, preds, average=None, labels=[0, 1, 2], zero_division=0)
    return {
        "macro_f1": float(mf1),
        "weighted_f1": float(wf1),
        "accuracy": float(acc),
        "f1_clean": float(f1[0]),
        "f1_offensive": float(f1[1]),
        "f1_hate": float(f1[2])
    }

# 6. Model Training: A/B Controlled Experiment
- **Mô hình A (Baseline):** Đúng cấu hình gốc của tác giả bài báo (Standard Cross-Entropy).
- **Mô hình B (Proposed):** Mô hình có trọng số lớp (Weighted Cross-Entropy).

In [8]:
import inspect

def get_warmup_kwargs(train_dataset, batch_size, epochs, warmup_ratio=0.1):
    params = inspect.signature(TrainingArguments.__init__).parameters
    if 'warmup_ratio' in params:
        return {'warmup_ratio': warmup_ratio}
    gradient_accumulation_steps = 1
    total_steps = int(np.ceil(len(train_dataset) / (batch_size * gradient_accumulation_steps)) * epochs)
    warmup_steps = int(total_steps * warmup_ratio)
    print(f'Dùng warmup_steps={warmup_steps}.')
    return {'warmup_steps': warmup_steps}

# --- MÔ HÌNH A: BASELINE (STANDARD CROSS-ENTROPY) ---
seed_everything(SEED)
output_dir_A = os.path.join(CHECKPOINT_DIR, "phobert_baseline")

training_args_A = TrainingArguments(
    output_dir=output_dir_A,
    num_train_epochs=4,
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    weight_decay=0.01,
    **get_warmup_kwargs(train_dataset, batch_size=16, epochs=4, warmup_ratio=0.1),
    max_grad_norm=1.0,
    **{EVAL_STRATEGY_PARAM: "epoch"},
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    greater_is_better=True,
    fp16=torch.cuda.is_available(),
    logging_steps=100,
    save_total_limit=1,
    seed=SEED,
    report_to="none"
)

print("Khởi tạo Mô hình A (Baseline)...")
model_A = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=3)
trainer_A = WeightedLossTrainer(
    model=model_A,
    args=training_args_A,
    train_dataset=train_dataset,
    eval_dataset=dev_dataset,
    compute_metrics=compute_metrics,
    data_collator=data_collator,
    class_weights=None
)

print("🚀 Bắt đầu huấn luyện Mô hình A...")
trainer_A.train()
dev_res_A = trainer_A.evaluate(dev_dataset)
print(f"Dev Macro F1 (Model A): {dev_res_A['eval_macro_f1']:.4f}")


In [ ]:
# --- MÔ HÌNH B: PROPOSED (WEIGHTED CROSS-ENTROPY) ---
del model_A
if torch.cuda.is_available():
    torch.cuda.empty_cache()

seed_everything(SEED)
output_dir_B = os.path.join(CHECKPOINT_DIR, "phobert_classweight")

training_args_B = TrainingArguments(
    output_dir=output_dir_B,
    num_train_epochs=4,
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    weight_decay=0.01,
    **get_warmup_kwargs(train_dataset, batch_size=16, epochs=4, warmup_ratio=0.1),
    max_grad_norm=1.0,
    **{EVAL_STRATEGY_PARAM: "epoch"},
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    greater_is_better=True,
    fp16=torch.cuda.is_available(),
    logging_steps=100,
    save_total_limit=1,
    seed=SEED,
    report_to="none"
)

print("Khởi tạo Mô hình B (Class Weight)...")
model_B = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=3)
trainer_B = WeightedLossTrainer(
    model=model_B,
    args=training_args_B,
    train_dataset=train_dataset,
    eval_dataset=dev_dataset,
    compute_metrics=compute_metrics,
    data_collator=data_collator,
    class_weights=class_weights_tensor
)

print("🚀 Bắt đầu huấn luyện Mô hình B...")
trainer_B.train()
dev_res_B = trainer_B.evaluate(dev_dataset)
print(f"Dev Macro F1 (Model B): {dev_res_B['eval_macro_f1']:.4f}")


# 7. Evaluation & Side-by-side Confusion Matrix
Dự đoán trên tập Test (6,680 mẫu) và đối chiếu trực quan ma trận nhầm lẫn giữa Mô hình A và Mô hình B.

In [ ]:
# Dự đoán trên tập Test
print("Đang chạy dự đoán trên 6,680 mẫu Test...")
preds_A = np.argmax(trainer_A.predict(test_dataset).predictions, axis=-1)
preds_B = np.argmax(trainer_B.predict(test_dataset).predictions, axis=-1)

target_names = ["CLEAN (0)", "OFFENSIVE (1)", "HATE (2)"]

print("=" * 65)
print("KẾT QUẢ TEST - MÔ HÌNH A (BASELINE)")
print("=" * 65)
print(classification_report(y_test, preds_A, target_names=target_names, digits=4))

print("=" * 65)
print("KẾT QUẢ TEST - MÔ HÌNH B (PROPOSED - CLASS WEIGHT)")
print("=" * 65)
print(classification_report(y_test, preds_B, target_names=target_names, digits=4))

# Bảng so sánh Δ(B - A)
f1_a = f1_score(y_test, preds_A, average='macro', zero_division=0)
f1_b = f1_score(y_test, preds_B, average='macro', zero_division=0)
acc_a = accuracy_score(y_test, preds_A)
acc_b = accuracy_score(y_test, preds_B)
p_a, r_a, pc_a, _ = precision_recall_fscore_support(y_test, preds_A, average=None, labels=[0, 1, 2], zero_division=0)
p_b, r_b, pc_b, _ = precision_recall_fscore_support(y_test, preds_B, average=None, labels=[0, 1, 2], zero_division=0)

comparison_df = pd.DataFrame([
    {"Chỉ số": "Macro F1", "Model A": f"{f1_a:.4f}", "Model B": f"{f1_b:.4f}", "Chênh lệch Δ(B - A)": f"{f1_b - f1_a:+.4f}"},
    {"Chỉ số": "Accuracy", "Model A": f"{acc_a:.4f}", "Model B": f"{acc_b:.4f}", "Chênh lệch Δ(B - A)": f"{acc_b - acc_a:+.4f}"},
    {"Chỉ số": "F1-CLEAN", "Model A": f"{pc_a[0]:.4f}", "Model B": f"{pc_b[0]:.4f}", "Chênh lệch Δ(B - A)": f"{pc_b[0] - pc_a[0]:+.4f}"},
    {"Chỉ số": "F1-OFFENSIVE", "Model A": f"{pc_a[1]:.4f}", "Model B": f"{pc_b[1]:.4f}", "Chênh lệch Δ(B - A)": f"{pc_b[1] - pc_a[1]:+.4f}"},
    {"Chỉ số": "F1-HATE", "Model A": f"{pc_a[2]:.4f}", "Model B": f"{pc_b[2]:.4f}", "Chênh lệch Δ(B - A)": f"{pc_b[2] - pc_a[2]:+.4f}"},
    {"Chỉ số": "Recall-OFFENSIVE", "Model A": f"{r_a[1]:.4f}", "Model B": f"{r_b[1]:.4f}", "Chênh lệch Δ(B - A)": f"{r_b[1] - r_a[1]:+.4f}"},
    {"Chỉ số": "Recall-HATE", "Model A": f"{r_a[2]:.4f}", "Model B": f"{r_b[2]:.4f}", "Chênh lệch Δ(B - A)": f"{r_b[2] - r_a[2]:+.4f}"}
])
print("\n" + "=" * 65)
print("BẢNG ĐỐI CHIẾU SO SÁNH ĐỊNH LƯỢNG")
print("=" * 65)
print(comparison_df.to_string(index=False))

metrics_file = os.path.join(OUTPUT_DIR, "aggregated_metrics.csv")
comparison_df.to_csv(metrics_file, index=False, encoding='utf-8')
print(f"✅ Đã lưu bảng chỉ số vào: {metrics_file}")


In [ ]:
# Trực quan hóa Ma trận Nhầm lẫn Side-by-Side
cm_A = confusion_matrix(y_test, preds_A, labels=[0, 1, 2])
cm_B = confusion_matrix(y_test, preds_B, labels=[0, 1, 2])

labels_display = ["CLEAN", "OFFENSIVE", "HATE"]

fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))
sn.heatmap(cm_A, annot=True, fmt='d', cmap='Blues', cbar=False,
           xticklabels=labels_display, yticklabels=labels_display, ax=axes[0])
axes[0].set_title(f"Mô hình A: Baseline\nMacro F1: {f1_a:.4f}", fontsize=12, fontweight='bold')
axes[0].set_xlabel("Nhãn dự đoán")
axes[0].set_ylabel("Nhãn thực tế")

sn.heatmap(cm_B, annot=True, fmt='d', cmap='Oranges', cbar=False,
           xticklabels=labels_display, yticklabels=labels_display, ax=axes[1])
axes[1].set_title(f"Mô hình B: Proposed (Class Weight)\nMacro F1: {f1_b:.4f}", fontsize=12, fontweight='bold')
axes[1].set_xlabel("Nhãn dự đoán")
axes[1].set_ylabel("Nhãn thực tế")

plt.tight_layout()
cm_file = os.path.join(OUTPUT_DIR, "confusion_matrix.png")
plt.savefig(cm_file, dpi=300, bbox_inches='tight')
plt.show()
print(f"✅ Đã lưu ma trận nhầm lẫn vào: {cm_file}")


# 8. Export Deliverables: `predictions.csv`
Xuất file kết quả nộp bài chính thức của Mô hình B gồm đúng 4 cột: `id`, `text`, `ground_truth`, `predicted_label` theo đúng đặc tả đề bài.

In [ ]:
# Tạo file predictions.csv
predictions_df = pd.DataFrame({
    'id': test['id'].values,
    'text': test['free_text'].values,
    'ground_truth': [LABEL_NAMES[y] for y in y_test],
    'predicted_label': [LABEL_NAMES[y] for y in preds_B]
})

output_file = os.path.join(OUTPUT_DIR, "predictions.csv")
predictions_df.to_csv(output_file, index=False, encoding='utf-8')
print(f"✅ Đã xuất tệp nộp bài: {output_file}")

assert len(predictions_df) == 6680, "Kích thước phải đúng 6,680 dòng!"
assert list(predictions_df.columns) == ['id', 'text', 'ground_truth', 'predicted_label'], "Tên cột phải đúng!"
assert predictions_df['predicted_label'].isnull().sum() == 0, "Không được có giá trị null!"
print("🎉 KIỂM TOÁN HOÀN TẤT: File nộp bài đạt 100% chuẩn chất lượng!")
print(predictions_df.head())


---
# 9. Lưu trữ Kết quả & Mô hình về Google Drive
Chạy cell này sau khi huấn luyện xong để mount Google Drive và tự động đồng bộ toàn bộ:
- Tệp dự đoán nộp bài 
- Bảng chỉ số đối chứng 
- Biểu đồ ma trận nhầm lẫn 
- Trọng số mô hình tốt nhất (/) và 

In [ ]:
# =====================================================================
# LƯU TRỮ KẾT QUẢ VÀ MÔ HÌNH VỀ GOOGLE DRIVE (SEED42_TEENCODE_35)
# =====================================================================
import os
import shutil

try:
    from google.colab import drive
    print("Đang kết nối Google Drive...")
    drive.mount("/content/drive")
    DRIVE_PROJECT_DIR = "/content/drive/MyDrive/PhoBERT_ViHSD"
except Exception as e:
    print(f"Chạy cục bộ hoặc không dùng Colab Drive: {e}")
    DRIVE_PROJECT_DIR = os.path.abspath("./GoogleDrive_Backup")

DRIVE_OUTPUT_DIR = os.path.join(DRIVE_PROJECT_DIR, "outputs", "seed42_teencode_35")
DRIVE_CHECKPOINT_DIR = os.path.join(DRIVE_PROJECT_DIR, "checkpoints", "seed42_teencode_35")
DRIVE_MODEL_DIR = os.path.join(DRIVE_CHECKPOINT_DIR, "final_phobert_class_weight_model")

os.makedirs(DRIVE_OUTPUT_DIR, exist_ok=True)
os.makedirs(DRIVE_CHECKPOINT_DIR, exist_ok=True)
os.makedirs(DRIVE_MODEL_DIR, exist_ok=True)

# Sao chép kết quả
print(f"Đang đồng bộ kết quả từ {OUTPUT_DIR} sang Google Drive...")
if os.path.exists(OUTPUT_DIR):
    for fname in os.listdir(OUTPUT_DIR):
        src_file = os.path.join(OUTPUT_DIR, fname)
        dst_file = os.path.join(DRIVE_OUTPUT_DIR, fname)
        if os.path.isfile(src_file):
            shutil.copy2(src_file, dst_file)
            print(f"  ✅ Đã lưu: {fname} -> {DRIVE_OUTPUT_DIR}")

# Lưu mô hình Model B và Tokenizer
print(f"Đang lưu mô hình Model B và Tokenizer vào: {DRIVE_MODEL_DIR}...")
try:
    if "trainer_B" in locals():
        trainer_B.save_model(DRIVE_MODEL_DIR)
        tokenizer.save_pretrained(DRIVE_MODEL_DIR)
        print("  ✅ Đã lưu mô hình từ trainer_B thành công!")
    elif "model_B" in locals():
        model_B.save_pretrained(DRIVE_MODEL_DIR)
        tokenizer.save_pretrained(DRIVE_MODEL_DIR)
        print("  ✅ Đã lưu mô hình từ model_B thành công!")
except Exception as err:
    print(f"Lưu ý khi lưu model: {err}")

print("=" * 65)
print("🎉 ĐÃ LƯU TRỮ TOÀN BỘ KẾT QUẢ VÀ MÔ HÌNH VỀ GOOGLE DRIVE THÀNH CÔNG!")
print(f"📁 Thư mục kết quả Drive:  {DRIVE_OUTPUT_DIR}")
print(f"📁 Thư mục mô hình Drive:   {DRIVE_MODEL_DIR}")
print("=" * 65)
